<a href="https://colab.research.google.com/github/EugeneoTK/CP_RAG/blob/main/HSG_Care_Protocol_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchainhub langchain-community langchain-openai chromadb beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 

In [ ]:
import os
from google.colab import userdata
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from bs4 import BeautifulSoup

# 1. Set up the API Key
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 2. Load the website
url = "https://www.primarycarepages.sg/healthier-sg/care-protocols/chronic-care-protocols/" # Replace with your target URL
loader = RecursiveUrlLoader(
    url=url,
    max_depth=2,
    extractor=lambda x: BeautifulSoup(x, "html.parser").text
)
docs = loader.load()
print(f"Successfully loaded {len(docs)} pages from the site.")

# 3. Split the text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

print(f"Divided the website into {len(splits)} chunks.")

Successfully loaded 77 pages from the site.
Divided the website into 1902 chunks.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Initialize the "Translator" (Embeddings)
embeddings = OpenAIEmbeddings()

# 2. Create the "Knowledge Base" (Vector Store)
# This takes your splits, turns them into numbers, and saves them in memory
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

print("Knowledge base is ready!")

Knowledge base is ready!


In [ ]:
!pip install -U langchain-classic langchain-openai langchain-community

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 1. Setup the Components
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 2. Define the Prompt (The Instructions)
template = """Answer the question based only on the following context.
If you don't know, say you don't know.

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 3. Create the Answer Logic
# This part turns context + question into an answer string
answer_chain = prompt | llm | StrOutputParser()

# 4. Create the Final "Parallel" Chain
# This runs the retriever AND the answer logic together so we keep the source docs
full_rag_chain = RunnableParallel({
    "context": retriever,
    "question": RunnablePassthrough()
}).assign(answer=answer_chain)

# 5. Run and Print with Citations
query = "When was FH Genetic Testing service introduced and for what condition does it applies to"
result = full_rag_chain.invoke(query)

print(f"🤖 ANSWER:\n{result['answer']}\n")
print("📍 SOURCES USED:")
# We use a set() to avoid printing the same URL multiple times
unique_sources = set(doc.metadata['source'] for doc in result['context'])
for source in unique_sources:
    print(f"- {source}")

🤖 ANSWER:
The Familial Hypercholesterolaemia (FH) Genetic Testing service was introduced on 30 June 2025. It applies to the condition of Familial Hypercholesterolaemia, which is an inherited genetic condition causing impaired LDL-cholesterol (LDL-C) metabolism and predisposes individuals to early onset atherosclerotic cardiovascular disease (ASCVD).

📍 SOURCES USED:
- https://www.primarycarepages.sg/healthier-sg/care-protocols/chronic-care-protocols/lipid-disorders
- https://www.primarycarepages.sg/schemes-and-programmes/familial-hypercholesterolaemia-genetic-testing-programme
